# Deepfake Detection — EfficientNet-B4 Fine-Tuning

## Before you start
1. `Runtime → Change runtime type → T4 GPU → Save`
2. Place `Celeb-DF-v2.zip` at: `Google Drive / deepfake_model / data / Celeb-DF-v2.zip`

## How to use
| Situation | What to do |
|---|---|
| **First ever run** | Run Cell A (install) → restart → run Cell B → run Cell C → leave |
| **Session restarted / came back** | Run Cell B → run Cell C (resumes from last checkpoint) |
| **Just want plots / export** | Run Cell B → run Cell D or E |

---
**Cell C runs the full pipeline unattended:** extract frames → crop faces → train → export ONNX.
Every step checks if it already finished, so re-running never repeats completed work.

---
## Cell A — Install (run once, then restart session)

In [ ]:
!pip install -q timm onnx onnxruntime scikit-learn
print('Done — Runtime → Restart session')

---
## Cell B — Setup (run every session before anything else)

In [ ]:
# ── 0. Install packages (safe to re-run every session) ───────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'timm', 'onnx', 'onnxruntime', 'scikit-learn'], check=True)

# ── 1. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── 2. Keep-alive: inject JavaScript into the browser ────────────────────────
from IPython.display import Javascript, display
display(Javascript("""
    if (window._dfKeepalive) clearInterval(window._dfKeepalive);
    window._dfKeepalive = setInterval(() => {
        var btn = document.querySelector('colab-toolbar-button#connect');
        if (btn) btn.click();
        console.log('[keep-alive]', new Date().toLocaleTimeString());
    }, 55000);
    console.log('[keep-alive] started');
"""))

# ── 3. Imports ────────────────────────────────────────────────────────────────
import os, json, time, datetime, zipfile, warnings
import numpy as np
import cv2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import timm
import onnx
import onnxruntime as ort
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

# ── 4. Config ─────────────────────────────────────────────────────────────────
DRIVE_BASE       = '/content/drive/MyDrive/deepfake_model'
ZIP_PATH         = f'{DRIVE_BASE}/data/Celeb-DF-v2.zip'
EXTRACT_DIR      = '/content/dataset'           # temp — re-extracted each run (fast)
FRAMES_DIR       = f'{DRIVE_BASE}/frames'       # Drive — survives any disconnect
PROCESSED_DIR    = f'{DRIVE_BASE}/processed'    # Drive — survives any disconnect
SAVE_DIR         = f'{DRIVE_BASE}/checkpoints'  # Drive — survives any disconnect

IMG_SIZE         = 224
BATCH_SIZE       = 32
NUM_EPOCHS       = 20
LR               = 1e-4
FRAMES_PER_VIDEO = 10
DEVICE           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for d in [SAVE_DIR, EXTRACT_DIR,
          f'{FRAMES_DIR}/real',   f'{FRAMES_DIR}/fake',
          f'{PROCESSED_DIR}/real', f'{PROCESSED_DIR}/fake']:
    os.makedirs(d, exist_ok=True)

# ── 5. Verify GPU ─────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), 'No GPU! Runtime → Change runtime type → T4 GPU'
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'PyTorch : {torch.__version__}')
print(f'ZIP     : {"FOUND" if os.path.isfile(ZIP_PATH) else "NOT FOUND — check Drive path"}')
print(f'Frames  : {len(list(Path(FRAMES_DIR).rglob("*.jpg")))} already on Drive')
print(f'Crops   : {len(list(Path(PROCESSED_DIR).rglob("*.jpg")))} already on Drive')
print()
print('Cell B done. Run Cell C to start full pipeline.')

---
## Cell C — Full pipeline (run and leave)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FULL PIPELINE — runs everything unattended:
#   1. Extract dataset
#   2. Extract frames
#   3. Detect & crop faces
#   4. Train EfficientNet-B4
#   5. Export to ONNX
#
# Every step checks if already done — safe to re-run after a disconnect.
# Checkpoints saved to Drive after every epoch.
# ════════════════════════════════════════════════════════════════════════════

import threading

pipeline_start = time.time()

def _log(msg):
    ts = datetime.datetime.now().strftime('%H:%M:%S')
    print(f'[{ts}] {msg}', flush=True)


# ── Step 1: Extract ZIP ───────────────────────────────────────────────────────
_log('Step 1/5 — Extract dataset')
if not os.path.isdir(f'{EXTRACT_DIR}/Celeb-real'):
    assert os.path.isfile(ZIP_PATH), f'ZIP not found at {ZIP_PATH}'
    _log(f'  Extracting {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    _log('  Extraction complete')
else:
    _log('  Already extracted, skipping')

root = Path(EXTRACT_DIR)
real_videos = list((root/'Celeb-real').glob('*.mp4')) + \
              list((root/'YouTube-real').glob('*.mp4'))
fake_videos = list((root/'Celeb-synthesis').glob('*.mp4'))
_log(f'  Real videos: {len(real_videos)} | Fake videos: {len(fake_videos)}')


# ── Step 2: Extract frames ────────────────────────────────────────────────────
_log('Step 2/5 — Extract frames')

def extract_frames(video_path, output_dir, n_frames=10):
    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release(); return 0
    indices = np.linspace(0, total-1, min(n_frames, total), dtype=int)
    stem    = Path(video_path).stem
    saved   = 0
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret: continue
        cv2.imwrite(str(Path(output_dir)/f'{stem}_f{idx:06d}.jpg'), frame)
        saved += 1
    cap.release()
    return saved

real_fr = Path(FRAMES_DIR)/'real'
fake_fr = Path(FRAMES_DIR)/'fake'
real_fr.mkdir(parents=True, exist_ok=True)
fake_fr.mkdir(parents=True, exist_ok=True)

if len(list(real_fr.glob('*.jpg'))) < len(real_videos) * 5:
    _log('  Extracting real frames...')
    for v in tqdm(real_videos, desc='real frames'):
        extract_frames(v, real_fr, FRAMES_PER_VIDEO)
else:
    _log('  Real frames already extracted, skipping')

if len(list(fake_fr.glob('*.jpg'))) < len(fake_videos) * 5:
    _log('  Extracting fake frames...')
    for v in tqdm(fake_videos, desc='fake frames'):
        extract_frames(v, fake_fr, FRAMES_PER_VIDEO)
else:
    _log('  Fake frames already extracted, skipping')

_log(f'  Frames done — real:{len(list(real_fr.glob("*.jpg")))} fake:{len(list(fake_fr.glob("*.jpg")))}')


# ── Step 3: Face crop ─────────────────────────────────────────────────────────
_log('Step 3/5 — Detect & crop faces')

try:
    from facenet_pytorch import MTCNN as _MTCNN
    _mtcnn     = _MTCNN(image_size=IMG_SIZE, margin=20, device=DEVICE,
                        keep_all=False, post_process=False)
    USE_MTCNN  = True
    _log('  Face detector: MTCNN')
except Exception as _e:
    USE_MTCNN  = False
    _cascade   = cv2.CascadeClassifier(
        cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    _log(f'  Face detector: OpenCV cascade (MTCNN unavailable: {_e})')

def crop_face(img_path, out_path):
    img = cv2.imread(str(img_path))
    if img is None: return False
    if USE_MTCNN:
        pil = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        try:
            face = _mtcnn(pil)
            if face is not None:
                arr = face.permute(1,2,0).cpu().numpy().astype(np.uint8)
                cv2.imwrite(str(out_path), cv2.cvtColor(arr, cv2.COLOR_RGB2BGR))
                return True
        except Exception:
            pass
    # fallback
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = [] if USE_MTCNN else \
            _cascade.detectMultiScale(gray, 1.1, 4, minSize=(50,50))
    if len(faces):
        x,y,w,h = faces[0]
        m        = int(max(w,h)*0.2)
        crop     = img[max(0,y-m):min(img.shape[0],y+h+m),
                       max(0,x-m):min(img.shape[1],x+w+m)]
    else:
        H,W  = img.shape[:2]; s = min(H,W)
        crop = img[(H-s)//2:(H+s)//2, (W-s)//2:(W+s)//2]
    cv2.imwrite(str(out_path), cv2.resize(crop, (IMG_SIZE,IMG_SIZE)))
    return True

real_out = Path(PROCESSED_DIR)/'real'
fake_out = Path(PROCESSED_DIR)/'fake'

if len(list(real_out.glob('*.jpg'))) < len(list(real_fr.glob('*.jpg'))) * 0.9:
    _log('  Cropping real faces...')
    for p in tqdm(list(real_fr.glob('*.jpg')), desc='crop real'):
        crop_face(p, real_out/p.name)
else:
    _log('  Real faces already cropped, skipping')

if len(list(fake_out.glob('*.jpg'))) < len(list(fake_fr.glob('*.jpg'))) * 0.9:
    _log('  Cropping fake faces...')
    for p in tqdm(list(fake_fr.glob('*.jpg')), desc='crop fake'):
        crop_face(p, fake_out/p.name)
else:
    _log('  Fake faces already cropped, skipping')

n_real = len(list(real_out.glob('*.jpg')))
n_fake = len(list(fake_out.glob('*.jpg')))
_log(f'  Crops done — real:{n_real} fake:{n_fake}')


# ── Step 4: Train ─────────────────────────────────────────────────────────────
_log('Step 4/5 — Build dataset & train')

class DeepfakeDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.tf = paths, labels, transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert('RGB')), self.labels[i]

all_paths  = [str(p) for p in real_out.glob('*.jpg')] + \
             [str(p) for p in fake_out.glob('*.jpg')]
all_labels = [0]*n_real + [1]*n_fake

tr_p,tmp_p,tr_l,tmp_l = train_test_split(all_paths, all_labels,
                                          test_size=0.2, stratify=all_labels, random_state=42)
va_p,te_p,va_l,te_l   = train_test_split(tmp_p, tmp_l,
                                          test_size=0.5, stratify=tmp_l, random_state=42)
_log(f'  Split — train:{len(tr_p)} val:{len(va_p)} test:{len(te_p)}')

MEAN,STD  = [0.485,0.456,0.406],[0.229,0.224,0.225]
train_tf  = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2,0.2,0.1,0.05),
    transforms.RandomAffine(degrees=10, translate=(0.05,0.05)),
    transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
eval_tf   = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize(MEAN,STD)])

sw          = [1.0/[tr_l.count(0),tr_l.count(1)][l] for l in tr_l]
sampler     = WeightedRandomSampler(sw, len(sw))
train_loader= DataLoader(DeepfakeDataset(tr_p,tr_l,train_tf),
                         BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader  = DataLoader(DeepfakeDataset(va_p,va_l,eval_tf),
                         BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(DeepfakeDataset(te_p,te_l,eval_tf),
                         BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

model     = timm.create_model('efficientnet_b4', pretrained=True, num_classes=2).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

def _train_ep(m,ld,opt,crit):
    m.train(); ls,cor,tot=0.,0,0
    for x,y in ld:
        x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad()
        o=m(x); l=crit(o,y); l.backward(); opt.step()
        ls+=l.item()*x.size(0); cor+=(o.argmax(1)==y).sum().item(); tot+=x.size(0)
    return ls/tot, cor/tot

def _eval_ep(m,ld,crit):
    m.eval(); ls,cor,tot=0.,0,0; pa,la=[],[]
    with torch.no_grad():
        for x,y in ld:
            x,y=x.to(DEVICE),y.to(DEVICE); o=m(x); l=crit(o,y)
            ls+=l.item()*x.size(0); cor+=(o.argmax(1)==y).sum().item(); tot+=x.size(0)
            pa.extend(torch.softmax(o,1)[:,1].cpu().numpy()); la.extend(y.cpu().numpy())
    return ls/tot, cor/tot, roc_auc_score(la,pa)

history   = {'train_loss':[],'train_acc':[],'val_loss':[],'val_acc':[],'val_auc':[]}
best_auc  = 0.0
start_ep  = 0

resume = f'{SAVE_DIR}/best_model.pth'
if os.path.isfile(resume):
    ck = torch.load(resume, map_location=DEVICE)
    model.load_state_dict(ck['model_state_dict'])
    optimizer.load_state_dict(ck['optimizer_state_dict'])
    scheduler.load_state_dict(ck['scheduler_state_dict'])
    start_ep = ck['epoch']+1; best_auc = ck['val_auc']
    if os.path.isfile(f'{SAVE_DIR}/history.json'):
        history = json.load(open(f'{SAVE_DIR}/history.json'))
    _log(f'  Resumed from epoch {start_ep} | best AUC: {best_auc:.4f}')

_log(f'  Training {NUM_EPOCHS} epochs on {torch.cuda.get_device_name(0)}')
print('-'*70)

t0 = time.time()
for ep in range(start_ep, NUM_EPOCHS):
    t1 = time.time()
    tl,ta = _train_ep(model, train_loader, optimizer, criterion)
    vl,va,vauc = _eval_ep(model, val_loader, criterion)
    scheduler.step()
    history['train_loss'].append(tl); history['train_acc'].append(ta)
    history['val_loss'].append(vl);   history['val_acc'].append(va)
    history['val_auc'].append(vauc)
    is_best = vauc > best_auc
    if is_best: best_auc = vauc
    state = {'epoch':ep,'model_state_dict':model.state_dict(),
             'optimizer_state_dict':optimizer.state_dict(),
             'scheduler_state_dict':scheduler.state_dict(),
             'val_acc':va,'val_auc':vauc}
    torch.save(state, f'{SAVE_DIR}/checkpoint_ep{ep:02d}.pth')
    if is_best: torch.save(state, f'{SAVE_DIR}/best_model.pth')
    with open(f'{SAVE_DIR}/history.json','w') as f: json.dump(history,f,indent=2)
    elapsed = str(datetime.timedelta(seconds=int(time.time()-t0)))
    print(f"Ep {ep+1:02d}/{NUM_EPOCHS} "
          f"| tr loss={tl:.4f} acc={ta:.3f} "
          f"| va loss={vl:.4f} acc={va:.3f} AUC={vauc:.4f} "
          f"| {int(time.time()-t1)}s | {elapsed}"
          + (" ★" if is_best else ""), flush=True)

_log(f'  Training done. Best AUC: {best_auc:.4f}')


# ── Step 5: ONNX export ───────────────────────────────────────────────────────
_log('Step 5/5 — Export to ONNX')
ck = torch.load(f'{SAVE_DIR}/best_model.pth', map_location=DEVICE)
model.load_state_dict(ck['model_state_dict'])
model.eval()

onnx_path = f'{SAVE_DIR}/deepfake_detector.onnx'
torch.onnx.export(model, torch.randn(1,3,IMG_SIZE,IMG_SIZE).to(DEVICE), onnx_path,
                  input_names=['input'], output_names=['output'],
                  dynamic_axes={'input':{0:'batch'},'output':{0:'batch'}},
                  opset_version=17)
onnx.checker.check_model(onnx.load(onnx_path))

sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
dummy_np = np.random.randn(1,3,IMG_SIZE,IMG_SIZE).astype(np.float32)
raw = sess.run(['output'], {'input': dummy_np})[0]
prob = np.exp(raw)/np.exp(raw).sum(axis=1,keepdims=True)
_log(f'  ONNX test — real:{prob[0][0]:.3f} fake:{prob[0][1]:.3f}')
_log(f'  Saved: {onnx_path}')

total_time = str(datetime.timedelta(seconds=int(time.time()-pipeline_start)))
print()
print('=' * 55)
print(f'  ALL DONE in {total_time}')
print(f'  Copy deepfake_detector.onnx into your app: models/')
print('=' * 55)

---
## Cell D — Training curves (run after training)

In [ ]:
import json, matplotlib.pyplot as plt
SAVE_DIR = '/content/drive/MyDrive/deepfake_model/checkpoints'

history = json.load(open(f'{SAVE_DIR}/history.json'))
fig, axes = plt.subplots(1, 3, figsize=(15,4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['val_acc'],   label='Val')
axes[1].set_title('Accuracy'); axes[1].legend()
axes[2].plot(history['val_auc'], color='purple')
axes[2].set_title('Val AUC')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150)
plt.show()
print(f'Best AUC: {max(history["val_auc"]):.4f} at epoch {history["val_auc"].index(max(history["val_auc"]))+1}')

---
## Cell E — Re-export ONNX only (if needed)

In [ ]:
import torch, timm, onnx, onnxruntime as ort, numpy as np
SAVE_DIR = '/content/drive/MyDrive/deepfake_model/checkpoints'
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224

model = timm.create_model('efficientnet_b4', pretrained=False, num_classes=2).to(DEVICE)
ck    = torch.load(f'{SAVE_DIR}/best_model.pth', map_location=DEVICE)
model.load_state_dict(ck['model_state_dict'])
model.eval()
print(f'Loaded checkpoint from epoch {ck["epoch"]+1}, val AUC={ck["val_auc"]:.4f}')

onnx_path = f'{SAVE_DIR}/deepfake_detector.onnx'
torch.onnx.export(model, torch.randn(1,3,IMG_SIZE,IMG_SIZE).to(DEVICE), onnx_path,
                  input_names=['input'], output_names=['output'],
                  dynamic_axes={'input':{0:'batch'},'output':{0:'batch'}},
                  opset_version=17)
onnx.checker.check_model(onnx.load(onnx_path))

sess    = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
raw     = sess.run(['output'], {'input': np.random.randn(1,3,IMG_SIZE,IMG_SIZE).astype(np.float32)})[0]
prob    = np.exp(raw)/np.exp(raw).sum(axis=1,keepdims=True)
print(f'ONNX verified — real:{prob[0][0]:.3f} fake:{prob[0][1]:.3f}')
print(f'Saved: {onnx_path}')